# Live demo — chat with our model on one business's memory (Colab, T4 or L4)

**Goal:** a chat link where anyone can type any question and get a real-time reply, using only this business's memory.
No server needed.

## How to run

1. Pick **T4** for practice or **L4** for the real demo, then **Run all**. Start **30 minutes early**: Cell 4 builds
   llama.cpp (~15 min).
2. Cell 7 runs the fixed test chats (normal questions + misuse attempts) and saves them for Claude to review.
3. Cell 8 prints a **public link** (`https://....gradio.live`) for the model chosen in `CHAT_MODEL`.
4. Keep this Colab tab open; the link stops when Colab stops. **Disconnect when done.**

To change the business details: edit Cell 5, then run Cells 5 to 8 again. Stop Cell 8 first.

## What happens to each message

1. The customer clearly mentions a complaint, refund, payment, or wants a person → hand-over (in their language).
2. Self-harm messages get a fixed caring reply with the Tele-MANAS helpline (14416).
3. Everything else goes to the model: greetings and small talk get a normal reply; off-topic and misuse (jokes,
   homework, prompt tricks, fake discounts, private data, abuse, politics, harmful requests) get a polite "I can only
   help with this shop"; questions the memory cannot answer get "I don't have that detail" + the phone.
4. A price, number, link, time or percentage that is not in the memory, or a discount / booking / free item the memory
   does not offer, becomes "I don't have that detail" + the phone. A reply that leaks the instructions or names the
   model becomes "I can only help with this shop".

Cell 8 prints the reason for every message.

In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive)
REPO_URL  = "https://github.com/ankit8453/chat-AI-model.git"   # private repo -> asks for a GitHub token once
REPO_PATH = "/content/chat-AI-model"

import os, subprocess, sys, getpass

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", url, REPO_PATH], capture_output=True, text=True)

if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "README.md")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    r = _clone(REPO_URL)
    if r.returncode != 0:
        print("Public clone failed (private repo?). Paste a GitHub token with read access:")
        tok = getpass.getpass("GitHub token: ")
        r = _clone(REPO_URL.replace("https://", f"https://{tok}@"))
        if r.returncode != 0:
            raise SystemExit("git clone failed:\n" + r.stderr[-1500:])
        subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", REPO_URL], check=False)
else:
    # Already cloned in this runtime -> pull the latest code (fixes pushed since the last run).
    r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], capture_output=True, text=True)
    if r.returncode != 0:
        tok = getpass.getpass("GitHub token (to pull updates): ")
        r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only",
                            REPO_URL.replace("https://", f"https://{tok}@"), "main"], capture_output=True, text=True)
    print("git pull:", (r.stdout or r.stderr).strip()[-300:])

print("code version:", subprocess.run(["git", "-C", REPO_PATH, "log", "-1", "--oneline"],
                                      capture_output=True, text=True).stdout.strip())
os.chdir(REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "src"))
print("Repo root contents:", sorted(os.listdir(".")))

proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24", "gradio>=5,<6"],
                      capture_output=True, text=True)
if proc.returncode != 0:
    print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
    raise SystemExit("pip install failed — see tail above.")
print("setup ok")

In [ ]:
# Cell 3 — HF Hub login
from huggingface_hub import login, HfApi
login()  # Colab inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
HF_NS = _whoami["name"]
print(f"HF Hub ok: user={HF_NS}")

In [ ]:
# Cell 4 — GPU check, download both models (ours + plain Qwen), build llama.cpp (~15 min the first time, no output until it finishes)
import os, shutil, subprocess, time, torch
from huggingface_hub import hf_hub_download
from serve.gguf import build_llama_cpp

subprocess.run(["nvidia-smi"], check=False)
EXPORT, LLAMA = "messages_2026-10-31", "/content/llama.cpp"
if not torch.cuda.is_available():
    raise SystemExit("No GPU — pick the T4 or L4 runtime.")

COMPARE = False          # True = also load plain Qwen 7B and run the test on both models
MODELS = {"mbg": "/content/models/mbg-qwen7b-Q4_K_M.gguf"}
os.makedirs("/content/models", exist_ok=True)
shutil.copy(hf_hub_download(f"{HF_NS}/mbg-qwen7b-gguf-{EXPORT}", "mbg-qwen7b-Q4_K_M.gguf"), MODELS["mbg"])
if COMPARE:
    MODELS["qwen"] = "/content/models/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf"
    for part in ("00001", "00002"):    # plain Qwen comes in two parts; llama.cpp loads them from the first
        hf_hub_download("Qwen/Qwen2.5-7B-Instruct-GGUF", f"qwen2.5-7b-instruct-q4_k_m-{part}-of-00002.gguf",
                        local_dir="/content/models")
t0 = time.time()
build_llama_cpp(LLAMA, "".join(map(str, torch.cuda.get_device_capability(0))))
print(f"models downloaded: {list(MODELS)}, llama.cpp ready in {(time.time() - t0) / 60:.0f} min")

In [ ]:
# Cell 5 — the business memory (same fields as config/memory_schema.json; leave out anything unknown)
MEMORY = {
    "business_name": "Jabalpur Cycle",
    "category": "Cycle shop",
    "about": "Cycle shop (साइकिल की दुकान) at Damoh Naka, Jabalpur, selling cycles, kids cycles, spare parts, tyres and tubes, "
             "bike lights and engine oil. Brands: Hero, Hercules, Avon, Firefox and Ninety One.",
    "services": [
        {"name": "Hero Canon 14T", "price": "about ₹4,055", "details": "kids cycle"},
        {"name": "Hero Blast", "price": "about ₹4,399 to ₹4,799", "details": "kids cycle"},
        {"name": "Hero Canon 20T", "price": "about ₹4,855", "details": "kids cycle"},
        {"name": "Hero SuperHero 20T", "price": "about ₹5,580", "details": "kids cycle"},
        {"name": "Hero Whistle", "price": "about ₹5,330 to ₹6,155", "details": "kids cycle"},
        {"name": "Kids cycle sizes", "details": "14T for small kids, 20T for older kids"},
        {"name": "Hero Jet Gold", "price": "about ₹6,300 to ₹6,345", "details": "daily use roadster cycle"},
        {"name": "Hercules Commander 26T", "price": "about ₹5,985", "details": "daily use cycle"},
        {"name": "Hercules Redeem XR RF VB", "price": "about ₹7,335", "details": "daily use cycle"},
        {"name": "Hercules Storm ZX", "price": "about ₹7,941 to ₹8,026", "details": "daily use cycle"},
        {"name": "Hero Sprint", "price": "from about ₹8,999", "details": "daily use cycle, no gears"},
        {"name": "Ninety One Thunder 26T", "price": "about ₹9,149", "details": "daily use cycle"},
        {"name": "Hercules Shona 26T", "price": "about ₹7,065", "details": "ladies cycle"},
        {"name": "Hero Miss India Gold 24T", "price": "about ₹7,320", "details": "ladies cycle"},
        {"name": "Ninety One Granite 26T Women", "price": "about ₹11,293", "details": "ladies cycle"},
        {"name": "Firefox Breeze 24", "price": "about ₹12,900", "details": "ladies cycle"},
        {"name": "Hercules Top Gear Rift 24SP", "price": "about ₹13,306 to ₹14,107", "details": "gear cycle"},
        {"name": "Hercules Roadeo Beast 21SP", "price": "about ₹18,505 to ₹19,573", "details": "gear cycle"},
        {"name": "Cycle chains, pedals and handles", "details": "spare parts"},
        {"name": "Tyres and tubes", "details": "for cycles and bikes"},
        {"name": "Bike lights"},
        {"name": "Engine oil"},
        {"name": "Pearl tyre tube repair patch", "price": "₹63 per piece"},
        {"name": "Cycle repair and service", "details": "at the shop; the charge depends on the work and is told after checking the cycle"},
    ],
    "timings": "Sunday to Friday 10:00 AM to 9:00 PM, Saturday closed",
    "locations": [{"address": "Shop No. 8, Gopal Sadan, Damoh Naka", "city": "Jabalpur, Madhya Pradesh 482002"}],
    "contact": {"phone": "7415145014", "whatsapp": "7415145014"},
    "faqs": [
        {"question": "Are these prices final?", "answer": "These are approximate prices. The final price is confirmed at the shop."},
        {"question": "Which cycle is good for a small child?", "answer": "14T cycles are for small kids, and 20T cycles are for older kids."},
        {"question": "Do you have gear cycles?", "answer": "Yes, from Hercules, starting around ₹13,300."},
        {"question": "Do you sell tyres and tubes separately?", "answer": "Yes."},
        {"question": "Is a test ride possible?", "answer": "Yes, at the shop."},
    ],
    "policies": {
        "delivery": "Home delivery in Jabalpur can be arranged, and the charge depends on the area.",
        "payment": "Cash and UPI accepted.",
        "EMI": "Not available for now.",
        "warranty": "As per the brand's terms, with a bill given on every new cycle.",
        "exchange": "Old cycle exchange is decided at the shop after checking the cycle.",
        "assembly": "New cycles are given fully fitted.",
    },
}
from sft.build import memory_chunks
for c in memory_chunks(MEMORY):
    print(f"[{c['type']}] {c['text']}\n")

In [ ]:
# Cell 6 — start the model(s) (ours on port 8765, plain Qwen on 8766 if COMPARE) + one message each
from demo.chat import answer
from mbg_eval.generate import generate_replies_http
from serve.gguf import start_server

for s in globals().get("servers", {}).values():
    s.terminate(); s.wait()
PORTS = {"mbg": 8765, "qwen": 8766}
servers = {m: start_server(LLAMA, MODELS[m], port=PORTS[m], log_path=f"/content/llama_{m}.log") for m in MODELS}
PORTS = {m: PORTS[m] for m in MODELS}

def asker(port):
    return lambda messages: generate_replies_http(f"http://127.0.0.1:{port}", [{"messages": messages}], workers=1)[0]

asks = {m: asker(p) for m, p in PORTS.items()}
for m, ask in asks.items():
    r = answer(MEMORY, "hii", [], ask)
    print(f"{m}: {r['text']!r}  [{r['reason']}]")

In [ ]:
# Cell 7 — the fixed test chats: normal questions + misuse attempts (saved to HF for Claude to review)
import json, tempfile, time
from huggingface_hub import HfApi
from demo.checks import run_checks

t0 = time.time()
rows = run_checks(MEMORY, asks)
for r in rows:
    if r["model"] == "mbg":
        print(f"\n[{r['kind']}] CUSTOMER: {r['message']}")
    flag = "  HAND-OVER" if r["needs_human"] else ""
    print(f"  {r['model']:<4}: {r['text'][:300]!r}  [{r['reason']}]{flag}")

name = f"demo_checks_{time.strftime('%Y%m%d_%H%M')}.jsonl"
path = f"{tempfile.mkdtemp()}/{name}"
with open(path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(json.dumps(r, ensure_ascii=False) for r in rows) + "\n")
HfApi().upload_file(path_or_fileobj=path, path_in_repo=name, repo_id=f"{HF_NS}/mbg-eval-{EXPORT}", repo_type="dataset",
                    commit_message="demo test chats")
for m in asks:
    mine = [r for r in rows if r["model"] == m]
    print(f"\n{m}: {sum(r['needs_human'] for r in mine)} hand-overs, "
          f"{sum(r['reason'] != 'ok' and not r['needs_human'] for r in mine)} replaced by 'no detail', of {len(mine)} messages")
print(f"\nsaved as {name} ({(time.time() - t0) / 60:.1f} min) — tell Claude it is uploaded")

In [ ]:
# Cell 8 — the chat link (this cell keeps running while the demo is live; stop it to end the demo)
from demo.ui import build_app

CHAT_MODEL = "mbg"          # "mbg" = our model, "qwen" = plain Qwen; pick after Claude's review of Cell 7
build_app(MEMORY, asks[CHAT_MODEL]).launch(share=True, debug=True)